# NB08 — Optimizer comparison on Yelp (seeds 42, 7, 13, 99)

COA vs PSO vs random search vs TPE for the ANN, under identical conditions, on **Yelp**. Three notebooks share
this exact code and differ only in `DATASET`:

| Notebook | Dataset | Seeds |
|---|---|---|
| `08-optcmp-amazon` | Amazon | 42, 7, 13, 99 |
| `08-optcmp-yelp` | Yelp | 42, 7, 13, 99 |
| `08-optcmp-twitter` | Twitter | 42, 7, 13, 99 |

Each (dataset, seed) pair is one "block"; NB12 pools the 12 blocks (Friedman test + Holm-corrected Wilcoxon tests).

**Why this version (Decision/014).** The former `08a/08b/08c-optcmp-*` notebooks ran everything in one Python process
and were killed for running out of RAM (08b after 3.6 h): with TensorFlow 2.20 / Keras 3 every `model.fit()` leaves
~20 TensorFlow functions and ~8,000 Python objects behind that `clear_session()` cannot free, and those notebooks
trained ~650 models in one process. Here every unit — one search (72 models) or one final model — runs in a **fresh
worker process** that exits when the unit is saved, so memory cannot accumulate. On "GPU T4 x2" two workers run side
by side, one per GPU (the former notebooks used one GPU), which roughly halves the time.

**Protocol — unchanged from the former 08 notebooks (the worker code is copied from them, and gives identical
results):**
- **Fitness** as in NB02: single GPU, float32, batch 1024, ≤ 8 epochs, early-stopping patience 1 on a 10% slice of
  the training partition (true labels), score = positive-class F1 at 0.5 on the validation set.
- **Budget**: exactly `BUDGET` = 72 fitness evaluations for every optimizer (COA may stop mid-iteration).
- **Common random numbers**: evaluation *k* of every optimizer uses TF seed `seed*1000+k+1`; COA, PSO and random
  search start from the same 8 random positions; TPE's 10 random start-up trials equal random search's first 10.
- **Final protocol** for every selected configuration and for the default (128/64/1e-3): retrain on train+validation
  (≤ 100 epochs, patience 10, true labels), evaluate once on the test set.

**Units:** per seed, 4 searches (COA, PSO, RS, TPE) and 5 final models (DEFAULT, COA, PSO, RS, TPE); a final model
starts after its search. Every unit is saved as soon as it finishes.

**Inputs:** `01-preprocessing` (required). Optional: the units that the former `08b-optcmp-yelp` finished before it ran
out of memory. Kaggle does not let you add a failed notebook version as an input, so: open that version → *Output* →
*Download All* → upload the zip unchanged as a private Kaggle Dataset (Kaggle unpacks it and keeps the folders) → add
that dataset here. Every linked input is searched for `saved_results/optcmp/yelp/`; matching finished units
are copied and not recomputed (the log says `Reused N finished unit(s)`). **Accelerator:** GPU T4 x2. **Internet:**
only needed if `optuna` is not preinstalled. **Runtime:** about 3.3 h (about 50 min per seed with two workers);
less when units are reused. Worst case, if every model trained to its epoch limit (never seen so far): about 8 h —
no new unit starts after `TIME_LIMIT_HOURS` = 8, every finished unit is already saved, and a rerun that gets this
run's output as an input (same download → dataset route) finishes the rest.
**Outputs:** `saved_results/optcmp/yelp/search_{OPT}_s{seed}.json`, `final_{OPT}_s{seed}.json`,
`summary_yelp.csv`, `fig_anytime_yelp.pdf`, worker logs in `logs/`, and the worker script
(`optcmp_worker.py`).

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATASET          = "yelp"
SEEDS            = [42, 7, 13, 99]   # one block per seed (the three 08 notebooks give NB12 12 blocks)
OPTIMIZERS       = ["COA", "PSO", "RS", "TPE"]
BUDGET           = 72          # fitness evaluations per search, identical for every optimizer
FINAL_EPOCHS     = 100
FINAL_PATIENCE   = 10
DEFAULT_CFG      = (128, 64, 1e-3)
N_PARALLEL       = None        # None = one worker per GPU (2 on "GPU T4 x2"); 1 = one worker at a time
TIME_LIMIT_HOURS = 8.0         # no new unit starts after this (Kaggle ends sessions at 9 h); normally ≈ 3.3 h
POLL_SECONDS     = 20          # how often worker logs are copied into this notebook's output

KAGGLE_USER = "abhaysharma01702"
NB01_SHARED = f"/kaggle/input/notebooks/{KAGGLE_USER}/01-preprocessing/shared_data/{DATASET}"
WORK        = "/kaggle/working"
RESULTS     = f"{WORK}/saved_results/optcmp/{DATASET}"
WORKER      = f"{WORK}/optcmp_worker.py"   # written by the next cell

## Worker script
The next cell only **writes** the worker to a file; the run cell below starts it once per unit, in a new process.

In [ ]:
%%writefile /kaggle/working/optcmp_worker.py
"""Optimizer-comparison worker for the 08-optcmp-<dataset> notebooks.

Runs ONE unit in a fresh Python process and exits:
  * a search: one optimizer (COA, PSO, RS or TPE), exactly BUDGET fitness evaluations, or
  * a final model: retrain the configuration a search selected (or the default 128/64/1e-3) on train+validation and
    evaluate it once on the test set.

Why a separate process per unit (Decision/014): with TensorFlow 2.20 / Keras 3, every model.fit() leaves ~20 TensorFlow
functions and ~8,000 Python objects behind that neither `del model`, `gc.collect()` nor `clear_session()` can free
(they come from the optimizer and metric variables created inside the compiled training step). The former 08a-c
notebooks trained ~650 models in one process and were killed for lack of RAM after ~3.6 h. A worker trains at most 72
models, and the operating system frees everything when it exits.

The computation (data pipeline, model, fitness, optimizers, final training, all seeds) is copied unchanged from the
former 08 template, so units computed by the former notebooks and by this worker are interchangeable.

Usage (the notebook calls it): python optcmp_worker.py '<json config>'
"""
import os, sys, json, time, gc, random as _random, warnings

warnings.filterwarnings("ignore")
CFG = json.loads(sys.argv[1])
TASK, DS, RUN_SEED, OPT = CFG["task"], CFG["dataset"], int(CFG["seed"]), CFG["optimizer"]
BUDGET, FINAL_EPOCHS, FINAL_PATIENCE = int(CFG["budget"]), int(CFG["final_epochs"]), int(CFG["final_patience"])
DEFAULT_CFG = tuple(CFG["default_cfg"])
D, R = CFG["shared_dir"], CFG["results_dir"]          # NB01 shared_data/<dataset>, saved_results/optcmp/<dataset>
TAG = f"[{OPT} s{RUN_SEED} {TASK}]"

import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")   # logging only: hides TF's retracing/deprecation notices (expected when many models are built)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef, roc_auc_score,
    average_precision_score, classification_report)
from sklearn.utils.class_weight import compute_class_weight

T0 = time.time()
SEED = 42
EMBED_DIM = 300

def rss_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**30
    except Exception:
        return float("nan")

def log(msg):
    print(f"{TAG} {msg}", flush=True)

# Single GPU, float32 — identical to the NB02 COA search (deliberately no mixed precision). The notebook makes one GPU
# visible to each worker (CUDA_VISIBLE_DEVICES), so two workers can run side by side on "GPU T4 x2".
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.set_visible_devices(gpus[0], "GPU")
    tf.config.experimental.set_memory_growth(gpus[0], True)
log(f"start: pid {os.getpid()}, TF {tf.__version__}, device {gpus[0].name if gpus else 'CPU'}, "
    f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', '-')}")

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed); np.random.seed(seed)

def set_seeds_tf(seed=SEED):
    set_seeds(seed); tf.random.set_seed(seed)

def save_json(path, obj, indent=2):
    """Atomic write: a process killed mid-write cannot leave a truncated result file behind."""
    with open(path + ".tmp", "w") as fh:
        json.dump(obj, fh, indent=indent)
    os.replace(path + ".tmp", path)

def load_json(path):
    with open(path) as fh:
        return json.load(fh)

def out(kind, opt):
    return f"{R}/{kind}_{opt}_s{RUN_SEED}.json"

# ── Shared pieces (NB02's model, decoding and search space) — unchanged ───────
ANN_BATCH = 1024
BOUNDS = np.array([[64, 256], [32, 128], [-4, -2]], dtype=float)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.float32)))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def build_ann(h1, h2, lr, seed):
    set_seeds_tf(seed)
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(h2, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

def decode(pos):
    return (int(round(np.clip(pos[0], 64, 256))),
            int(round(np.clip(pos[1], 32, 128))),
            float(10 ** np.clip(pos[2], -4, -2)))

def random_position(rng):
    return np.array([rng.uniform(lo, hi) for lo, hi in BOUNDS])

# ── Optimizers, each limited to exactly BUDGET evaluations — unchanged ────────
def coa_search(fitness, trace, run_seed, budget=BUDGET, n_coyotes=8, p_leave=0.1):
    """NB02's COA update rule, run until the evaluation budget is used up."""
    rng = np.random.RandomState(run_seed)
    pop  = [random_position(rng) for _ in range(n_coyotes)]
    fits = [fitness(p) for p in pop]
    while len(trace) < budget:
        alpha = pop[int(np.argmax(fits))].copy()
        tend  = np.median(pop, axis=0)
        for i in range(n_coyotes):
            if len(trace) >= budget: break
            if i == int(np.argmax(fits)): continue
            r1, r2 = rng.rand(), rng.rand()
            new_p = np.clip(pop[i] + r1 * (alpha - pop[i]) + r2 * (tend - pop[i]), BOUNDS[:, 0], BOUNDS[:, 1])
            new_f = fitness(new_p)
            if new_f > fits[i]:
                pop[i] = new_p; fits[i] = new_f
        if len(trace) < budget and rng.rand() < p_leave:
            wi = int(np.argmin(fits))
            cp = random_position(rng); cf = fitness(cp)
            if cf > fits[wi]: pop[wi] = cp; fits[wi] = cf
    bi = int(np.argmax(fits))
    return pop[bi], fits[bi]

def pso_search(fitness, trace, run_seed, budget=BUDGET, n_particles=8, w=0.5, c1=1.5, c2=1.5):
    """The original NB08 PSO, run until the evaluation budget is used up."""
    rng = np.random.RandomState(run_seed)
    dim = len(BOUNDS); ranges = BOUNDS[:, 1] - BOUNDS[:, 0]; v_max = ranges * 0.2
    positions  = np.array([random_position(rng) for _ in range(n_particles)])
    velocities = np.array([rng.uniform(-1, 1, dim) * ranges * 0.1 for _ in range(n_particles)])
    pbest_pos = positions.copy()
    pbest_fit = np.array([fitness(p) for p in positions])
    g = int(np.argmax(pbest_fit)); gbest_pos, gbest_fit = pbest_pos[g].copy(), pbest_fit[g]
    while len(trace) < budget:
        for i in range(n_particles):
            if len(trace) >= budget: break
            r1, r2 = rng.rand(dim), rng.rand(dim)
            velocities[i] = np.clip(w * velocities[i] + c1 * r1 * (pbest_pos[i] - positions[i])
                                    + c2 * r2 * (gbest_pos - positions[i]), -v_max, v_max)
            positions[i] = np.clip(positions[i] + velocities[i], BOUNDS[:, 0], BOUNDS[:, 1])
            f_ = fitness(positions[i])
            if f_ > pbest_fit[i]:
                pbest_fit[i] = f_; pbest_pos[i] = positions[i].copy()
                if f_ > gbest_fit: gbest_fit = f_; gbest_pos = positions[i].copy()
    return gbest_pos, gbest_fit

def rs_search(fitness, trace, run_seed, budget=BUDGET):
    rng = np.random.RandomState(run_seed)
    best_pos, best_fit = None, -1.0
    for _ in range(budget):
        pos = random_position(rng); f_ = fitness(pos)
        if f_ > best_fit: best_fit, best_pos = f_, pos.copy()
    return best_pos, best_fit

def tpe_search(fitness, trace, run_seed, budget=BUDGET):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        pos = np.array([trial.suggest_float("h1", 64, 256),
                        trial.suggest_float("h2", 32, 128),
                        trial.suggest_float("log10_lr", -4, -2)])
        return fitness(pos)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=run_seed))
    study.optimize(objective, n_trials=budget)
    bp = study.best_params
    return np.array([bp["h1"], bp["h2"], bp["log10_lr"]]), study.best_value

SEARCHERS = {"COA": coa_search, "PSO": pso_search, "RS": rs_search, "TPE": tpe_search}

# ── Data (NB01 outputs) ───────────────────────────────────────────────────────
X_emb_train, X_emb_val = (np.load(f"{D}/X_emb_{k}.npy") for k in ["train", "val"])
y_train, y_val = (np.load(f"{D}/y_{k}.npy") for k in ["train", "val"])
class_weights = {int(k): float(v) for k, v in json.load(open(f"{D}/class_weights.json")).items()}
log(f"data: train {X_emb_train.shape}, val {X_emb_val.shape}; RSS {rss_gb():.1f} GB")

if TASK == "search":
    if os.path.exists(out("search", OPT)):
        log("already finished — nothing to do"); sys.exit(0)
    # Fitness datasets, built exactly as in NB02 and the former 08 notebooks (true labels for early stopping).
    # They are created before any seed is set, as in the former notebooks, so the cached data order is identical.
    n_int  = int(len(y_train) * 0.1)
    fit_ds = make_tf_ds(X_emb_train[:-n_int], y_train[:-n_int], ANN_BATCH)
    iv_ds  = make_tf_ds(X_emb_train[-n_int:], y_train[-n_int:], ANN_BATCH, shuffle=False)
    val_ds = make_tf_ds(X_emb_val, np.zeros(len(y_val)), ANN_BATCH * 4, shuffle=False)   # predict only

    def make_fitness(run_seed, trace):
        """Evaluation k of every optimizer uses TF seed run_seed*1000 + k + 1 (common random numbers)."""
        def fitness(pos):
            k = len(trace); t_eval = time.time()
            h1, h2, lr = decode(pos)
            model = build_ann(h1, h2, lr, seed=run_seed * 1000 + k + 1)
            model.fit(fit_ds, validation_data=iv_ds, epochs=8, class_weight=class_weights,
                      callbacks=[EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True,
                                               verbose=0)], verbose=0)
            prob = model.predict(val_ds, verbose=0).flatten()
            f1 = float(f1_score(y_val, (prob > 0.5).astype(int)))
            trace.append({"k": k, "h1": h1, "h2": h2, "lr": lr, "f1": round(f1, 5)})
            del model; gc.collect(); tf.keras.backend.clear_session()
            best = max(t["f1"] for t in trace)
            log(f"eval {k + 1:>3}/{BUDGET}  H1={h1:<3} H2={h2:<3} LR={lr:.6f}  val F1={f1:.4f}  best={best:.4f}  "
                f"({time.time() - t_eval:.1f} s, RSS {rss_gb():.1f} GB)")
            return f1
        return fitness

    trace = []; t_task = time.time()
    best_pos, best_fit = SEARCHERS[OPT](make_fitness(RUN_SEED, trace), trace, RUN_SEED)
    assert len(trace) == BUDGET, f"{OPT} used {len(trace)} evaluations"
    h1, h2, lr = decode(best_pos)
    save_json(out("search", OPT),
              {"dataset": DS, "optimizer": OPT, "seed": RUN_SEED, "budget": BUDGET,
               "best_h1": h1, "best_h2": h2, "best_lr": lr, "best_val_f1": round(float(best_fit), 5),
               "best_so_far": [round(float(v), 5) for v in np.maximum.accumulate([t["f1"] for t in trace])],
               "trace": trace, "minutes": round((time.time() - t_task) / 60, 1), "code": "08-worker-2026-10-13"},
              indent=1)
    log(f"DONE  best val F1={best_fit:.4f}  H1={h1} H2={h2} LR={lr:.6f}  ({(time.time() - t_task) / 60:.1f} min, "
        f"RSS {rss_gb():.1f} GB)")

elif TASK == "final":
    if os.path.exists(out("final", OPT)):
        log("already finished — nothing to do"); sys.exit(0)
    X_emb_test, y_test = np.load(f"{D}/X_emb_test.npy"), np.load(f"{D}/y_test.npy")
    X_tv = np.vstack([X_emb_train, X_emb_val]); y_tv = np.concatenate([y_train, y_val])
    del X_emb_train, X_emb_val; gc.collect()
    n_es = int(len(y_tv) * 0.1)
    cw_tv = dict(zip([int(k) for k in np.unique(y_tv)],
                     [float(v) for v in compute_class_weight("balanced", classes=np.unique(y_tv), y=y_tv)]))

    def train_final(h1, h2, lr, run_seed):
        model = build_ann(h1, h2, lr, seed=run_seed)
        model.fit(make_tf_ds(X_tv[:-n_es], y_tv[:-n_es], ANN_BATCH, seed=run_seed),
                  validation_data=make_tf_ds(X_tv[-n_es:], y_tv[-n_es:], ANN_BATCH, shuffle=False),
                  epochs=FINAL_EPOCHS, class_weight=cw_tv,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE,
                                           restore_best_weights=True, verbose=0)], verbose=0)
        prob = model.predict(make_tf_ds(X_emb_test, np.zeros(len(y_test)), ANN_BATCH, shuffle=False),
                             verbose=0).flatten()
        del model; gc.collect(); tf.keras.backend.clear_session()
        pred = (prob > 0.5).astype(int)
        rep = classification_report(y_test, pred, output_dict=True, zero_division=0)
        return {"F1-score": round(f1_score(y_test, pred), 4), "Macro-F1": round(rep["macro avg"]["f1-score"], 4),
                "MCC": round(matthews_corrcoef(y_test, pred), 4), "AUC-ROC": round(roc_auc_score(y_test, prob), 4),
                "Accuracy": round(accuracy_score(y_test, pred), 4),
                "Avg-Precision": round(average_precision_score(y_test, prob), 4)}

    if OPT == "DEFAULT":
        (h1, h2, lr), val_f1 = DEFAULT_CFG, None
    else:
        sr = load_json(out("search", OPT))
        h1, h2, lr, val_f1 = sr["best_h1"], sr["best_h2"], sr["best_lr"], sr["best_val_f1"]
    t_task = time.time()
    res = train_final(h1, h2, lr, RUN_SEED)
    res.update({"dataset": DS, "optimizer": OPT, "seed": RUN_SEED, "h1": h1, "h2": h2, "lr": lr,
                "best_val_f1": val_f1, "minutes": round((time.time() - t_task) / 60, 1),
                "code": "08-worker-2026-10-13"})
    save_json(out("final", OPT), res)
    log(f"DONE  test F1={res['F1-score']}  MCC={res['MCC']}  H1={h1} H2={h2} LR={lr:.6f}  "
        f"({(time.time() - t_task) / 60:.1f} min, RSS {rss_gb():.1f} GB)")

else:
    raise ValueError(f"unknown task {TASK!r}")

## Run all units
Reuses finished units from linked inputs, then starts one worker process per unit — two at a time on T4 x2 — and
copies the workers' progress lines into this output. A failed unit does not stop the others.

In [ ]:
import os, sys, json, glob, shutil, subprocess, time, importlib.util
import numpy as np

T_START = time.time()
def hours_used():
    return (time.time() - T_START) / 3600

def load_json(path):
    with open(path) as fh:
        return json.load(fh)

def out(kind, opt, seed):
    return f"{RESULTS}/{kind}_{opt}_s{seed}.json"

if not os.path.isdir(NB01_SHARED):
    raise FileNotFoundError("Input missing: link '01-preprocessing' (Add Input → Your Work → Notebooks).")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(f"{WORK}/logs", exist_ok=True)
if "TPE" in OPTIMIZERS and importlib.util.find_spec("optuna") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)

# 1) Reuse units that an earlier run already finished (any linked input; only this dataset and these seeds). A search
#    is reused only if it used the same budget; a final model only if it was trained with exactly the configuration
#    that its search selected (the default configuration for DEFAULT).
def same_cfg(final, cfg):
    return final["h1"] == cfg[0] and final["h2"] == cfg[1] and bool(np.isclose(final["lr"], cfg[2], rtol=1e-9, atol=0))

found = sorted(glob.glob(f"/kaggle/input/**/saved_results/optcmp/{DATASET}/*_s*.json", recursive=True))
reused = []
for seed in SEEDS:
    for opt in OPTIMIZERS:
        if os.path.exists(out("search", opt, seed)):
            continue
        for f in [f for f in found if os.path.basename(f) == f"search_{opt}_s{seed}.json"]:
            d = load_json(f)
            if d.get("dataset") == DATASET and d.get("budget") == BUDGET and len(d.get("trace", [])) == BUDGET:
                shutil.copy(f, out("search", opt, seed)); reused.append(f); break
    for opt in OPTIMIZERS + ["DEFAULT"]:
        if os.path.exists(out("final", opt, seed)):
            continue
        if opt == "DEFAULT":
            cfg = DEFAULT_CFG
        elif os.path.exists(out("search", opt, seed)):
            s = load_json(out("search", opt, seed)); cfg = (s["best_h1"], s["best_h2"], s["best_lr"])
        else:
            continue
        for f in [f for f in found if os.path.basename(f) == f"final_{opt}_s{seed}.json"]:
            d = load_json(f)
            if d.get("dataset") == DATASET and same_cfg(d, cfg):
                shutil.copy(f, out("final", opt, seed)); reused.append(f); break
print(f"Reused {len(reused)} finished unit(s) from linked inputs:" if reused else "No finished units to reuse.")
for f in reused:
    print("   ", f.replace("\\", "/").split("/input/")[-1])

# 2) Start one worker process per unit; a final model waits for its search. Units are ordered seed by seed, so blocks
#    complete one after another.
def visible_gpus():
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=60)
        return [l for l in r.stdout.splitlines() if l.startswith("GPU ")]
    except Exception:
        return []

def ram_info():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f"RAM in use {(vm.total - vm.available) / 2**30:.1f} of {vm.total / 2**30:.1f} GB"
    except Exception:
        return "RAM: n/a"

GPUS = visible_gpus()
N_SLOTS = N_PARALLEL or max(1, len(GPUS))
UNITS = [(kind, opt, seed) for seed in SEEDS
         for kind, opt in [("search", o) for o in OPTIMIZERS] + [("final", "DEFAULT")] + [("final", o) for o in OPTIMIZERS]]
pending = [u for u in UNITS if not os.path.exists(out(*u))]
print(f"GPUs: {len(GPUS)}; {N_SLOTS} worker(s) at a time; {len(pending)} of {len(UNITS)} units to run; {ram_info()}")

# TensorFlow start-up messages that are not errors (the full worker logs in logs/ keep everything)
NOISE = ("Unable to register cu", "computation placer already registered", "All log messages before absl",
         "Compiled cluster using XLA", "gpu_device.cc", "cuda_executor.cc", "TF-TRT Warning",
         "oneDNN custom operations", "cpu_feature_guard", "rebuild TensorFlow", "gpu_timer.cc",
         "triggered tf.function retracing", "is deprecated. Please use")

def pump(r, final=False):
    """Copy the worker's new complete log lines into this output (everything left once the worker has ended)."""
    with open(r["log"], "rb") as fh:
        fh.seek(r["pos"]); data = fh.read()
    r["pos"] += len(data)
    parts = (r["buf"] + data).split(b"\n")
    r["buf"] = b"" if final else parts.pop()
    for raw in parts:
        line = raw.decode("utf-8", "replace").rstrip()
        if line and not any(n in line for n in NOISE):
            print("   ", line, flush=True)

def ready(u):
    kind, opt, seed = u
    return kind == "search" or opt == "DEFAULT" or os.path.exists(out("search", opt, seed))

def search_pending_or_running(opt, seed):
    u = ("search", opt, seed)
    return u in pending or any(r["unit"] == u for r in running.values())

running, failed, stopped, last_beat = {}, [], False, time.time()
while pending or running:
    for u in [u for u in pending if not ready(u) and not search_pending_or_running(u[1], u[2])]:
        pending.remove(u); failed.append((u, "its search did not finish"))   # can never become ready
    for slot in range(N_SLOTS):
        if slot in running or not pending:
            continue
        if hours_used() > TIME_LIMIT_HOURS:
            print(f"[{hours_used():.2f} h] Time limit reached: {len(pending)} unit(s) not started.", flush=True)
            stopped = True; pending.clear(); break
        nxt = next((u for u in pending if ready(u)), None)
        if nxt is None:
            break
        pending.remove(nxt)
        kind, opt, seed = nxt
        cfg = {"task": kind, "dataset": DATASET, "seed": seed, "optimizer": opt, "budget": BUDGET,
               "final_epochs": FINAL_EPOCHS, "final_patience": FINAL_PATIENCE, "default_cfg": list(DEFAULT_CFG),
               "shared_dir": NB01_SHARED, "results_dir": RESULTS}
        env = dict(os.environ)
        if GPUS:
            env["CUDA_VISIBLE_DEVICES"] = str(slot % len(GPUS))
        logf = f"{WORK}/logs/{DATASET}_s{seed}_{kind}_{opt}.log"
        with open(logf, "wb") as fh:
            proc = subprocess.Popen([sys.executable, "-u", WORKER, json.dumps(cfg)],
                                    stdout=fh, stderr=subprocess.STDOUT, env=env)
        running[slot] = {"unit": nxt, "proc": proc, "log": logf, "pos": 0, "buf": b"", "t0": time.time()}
        where = f"GPU {slot % len(GPUS)}" if GPUS else "CPU"
        print(f"[{hours_used():.2f} h] START {kind} {opt} s{seed} on {where}", flush=True)
    if not running:
        if pending:   # nothing can start (should not happen) — never wait forever
            failed += [(u, "could not be started") for u in pending]; pending.clear()
        break
    time.sleep(POLL_SECONDS)
    for slot, r in list(running.items()):
        pump(r)
        rc = r["proc"].poll()
        if rc is None:
            continue
        pump(r, final=True)
        kind, opt, seed = r["unit"]
        mins = (time.time() - r["t0"]) / 60
        if rc == 0 and os.path.exists(out(kind, opt, seed)):
            print(f"[{hours_used():.2f} h] END   {kind} {opt} s{seed}: ok ({mins:.1f} min)", flush=True)
        else:
            why = "killed by the system, most likely out of memory" if rc in (-9, 137) else f"exit code {rc}"
            print(f"[{hours_used():.2f} h] END   {kind} {opt} s{seed}: FAILED ({why}); full log: {r['log']}", flush=True)
            failed.append((r["unit"], why))
        del running[slot]
    if running and time.time() - last_beat > 600:
        names = ", ".join(f"{r['unit'][0]} {r['unit'][1]} s{r['unit'][2]}" for r in running.values())
        print(f"[{hours_used():.2f} h] running: {names}; {ram_info()}", flush=True)
        last_beat = time.time()

print(f"\nFinished after {hours_used():.2f} h. Failed units: {failed if failed else 'none'}"
      + ("  (stopped at the time limit)" if stopped else ""))

In [ ]:
# ── Summary over seeds and anytime curves ────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

rows = [load_json(out("final", o, s)) for s in SEEDS for o in OPTIMIZERS + ["DEFAULT"]
        if os.path.exists(out("final", o, s))]
if rows:
    df = pd.DataFrame(rows)
    print(df[["seed", "optimizer", "h1", "h2", "lr", "best_val_f1", "F1-score", "MCC", "AUC-ROC"]].to_string(index=False))
    agg = df.groupby("optimizer").agg(n=("seed", "count"),
                                      test_f1_mean=("F1-score", "mean"), test_f1_std=("F1-score", "std"),
                                      val_f1_mean=("best_val_f1", "mean"), val_f1_std=("best_val_f1", "std"))
    print(f"\n{DATASET.upper()} (pooled tests across datasets are in NB12)\n{agg.round(4).to_string()}")
    agg.round(5).to_csv(f"{RESULTS}/summary_{DATASET}.csv")

fig, ax = plt.subplots(figsize=(8, 4.5))
for o in OPTIMIZERS:
    curves = [load_json(out("search", o, s))["best_so_far"] for s in SEEDS if os.path.exists(out("search", o, s))]
    if curves:
        L = min(len(c) for c in curves)
        c = np.array([cv[:L] for cv in curves]); x = np.arange(1, L + 1)
        ax.plot(x, c.mean(0), label=f"{o} (n={len(curves)})")
        ax.fill_between(x, c.min(0), c.max(0), alpha=0.15)
ax.set_xlabel("Fitness evaluations"); ax.set_ylabel("Best validation F1 so far")
ax.set_title(f"Optimizer comparison — {DATASET.capitalize()} (mean; band = min–max over seeds)")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{RESULTS}/fig_anytime_{DATASET}.pdf", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# ── Status of the blocks ─────────────────────────────────────────────────────
missing = [f"{k} {o} s{s}" for k, o, s in UNITS if not os.path.exists(out(k, o, s))]
complete = [s for s in SEEDS if all(os.path.exists(out(k, o, s)) for k, o, s2 in UNITS if s2 == s)]
print(f"Complete blocks for {DATASET}: seeds {complete} of {SEEDS}.")
if not missing:
    print(f"All {len(SEEDS)} blocks COMPLETE ({len(UNITS)} units) in {RESULTS}.")
elif stopped and not failed:
    print(f"Stopped at the time limit with {len(missing)} unit(s) missing: {missing}.\n"
          "Finished units are saved in this version's output. To finish: Output → Download All → upload the zip "
          "unchanged as a private Kaggle Dataset → add it as an input and run this notebook again; finished units "
          "are reused automatically.")
else:
    raise RuntimeError(f"{DATASET}: INCOMPLETE. Failed: {failed}. Missing: {missing}. The worker logs in "
                       f"{WORK}/logs/ show why. Finished units are saved in this version's output; to reuse them, "
                       "download the output, upload it unchanged as a private Kaggle Dataset and add it as an input "
                       "of the rerun (a failed version cannot be linked directly).")